# Repeat 01 — did TRGT end up on the right haplotypes?

`TrgtHaplotypePhase` haplotags one sample's reads against the **ligated phased callset**, re-genotypes
the repeats in that window with TRGT, and joins read&nbsp;→&nbsp;haplotype against read&nbsp;→&nbsp;allele to put each
TRGT allele on a phased haplotype. This notebook checks whether that worked, and nothing else.

Window: `chr20:10,000,000-20,000,000`, the window `TrgtLocusConcordance` already ran, so the
integrated-callset comparison here is directly comparable to the concordance table behind Figure 2C.

Checks, in the order they can fail:

1. **Tag census and read accounting** — is the allele tag the one we passed, and were the reads
   actually haplotagged? An untagged BAM resolves nothing and looks exactly like a thin one.
2. **Resolution tally** — of the loci where orientation can matter at all (the two genotype slots
   hold different alleles), what fraction got a read-backed answer.
3. **Re-genotyping did not move the calls** — the unordered allele pair from the haplotag-aware
   re-run against the as-delivered production TRGT call, stratified by length change. A pair that
   changes is a genotyping difference, not a phasing one, and has to be understood first.
4. **Orientation against the integrated callset** — agreement should be high where our summed
   lengths are reliable (0 bp, 1–19 bp) and fall away at ≥ 50 bp. That fall is the expected result,
   not a failure: it is the double-counting this whole exercise exists to escape.
5. **Exposure** — how many person-loci are both length-heterozygous *and* ancestry-discordant.
   That product is the only place orientation changes a per-ancestry dosage, and it sizes whether
   the genome-wide version of this is worth running.

Nothing here writes to the manuscript data directory. This is a go/no-go on one sample.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

_bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
_sync = os.environ.get("TERRA_SYNC_SCRIPTS", "true" if _bucket else "").strip().lower() in {
    "1", "true", "yes", "on",
}
_scripts = None
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "aggregate_repeat_loci.py").is_file() or (_d / "terra_notebook.py").is_file():
        _scripts = _d.resolve()
        break
if _bucket and _sync:
    if _scripts is None:
        _scripts = (Path.cwd() / "scripts").resolve()
    _scripts.mkdir(parents=True, exist_ok=True)
    print(f"gsutil -m rsync -r {_bucket}/scripts/ {_scripts}/")
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_scripts) + "/"])
if _scripts and str(_scripts) not in sys.path:
    sys.path.insert(0, str(_scripts))
print("scripts:", _scripts)
for _name in ("phase_trgt_by_haplotag.py", "aggregate_repeat_loci.py"):
    assert (_scripts / _name).is_file(), f"{_name} not staged"

In [ ]:
import gzip
import hashlib
import json
import re
import shutil

import numpy as np
import pandas as pd

assert shutil.which("bcftools") and shutil.which("gsutil"), "bcftools / gsutil not on PATH"

BUCKET = "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09"
SAMPLE = "1000234"
REGION = "chr20:10000000-20000000"
CHROM, _span = REGION.split(":")
START, END = (int(x) for x in _span.split("-"))

WORK = Path("work/repeat01").resolve()
WORK.mkdir(parents=True, exist_ok=True)
REFS = f"{BUCKET}/refs/repeat_loci"
CATALOG = f"{REFS}/trexplorer_v1.0.1.catalog.bed.gz"

# TrgtHaplotypePhase outputs: the workflow directory, i.e. the run's own output
# URI cut at /call-HaplotagAndGenotype. This one is submission f2d6f6a7, the
# first WHOLE-GENOME run and the first on the Array[File] per-sample input.
# Its predecessor 201bc30d was windowed (chr20:10-20Mb) on corrected definitions.
#
# Note what this run did and did not cover: phased_vcfs held only the chr20
# per-sample VCF while region was "", so TRGT genotyped all 4,439,813 loci but
# whatshap had phase for chr20 alone. Resolution counts from it are chr20
# numbers over a genome-wide denominator -- frac_resolved_of_relevant reads
# 0.0203 for that reason and should not be quoted. Checks below that depend on
# resolution are therefore chr20-only until a run with all 22 VCFs exists.
RUN = os.environ.get("TRGT_PHASE_RUN", "").rstrip("/") or (
    f"{BUCKET}/submissions/f2d6f6a7-a8e7-4fe2-b701-9114a3bab32a"
    "/TrgtHaplotypePhase/68e97069-74c4-4945-8fef-3e1378633ac0"
)

# Integrated callset for the same window, for checks 4 and 5.
#
# This must be flare_lai_prod.annotated_vcf -- the PropagateFlareAncestry output
# -- and NOT the FLARE anc_vcf, because AN1/AN2 have to sit on the same records
# as the genotypes.
#
# Check 5 rests on hap1 <-> AN1: the ancestry haplotype order must be the order
# whatshap haplotagged in. That is NOT established by provenance. The workflow
# was configured with flare_lai_prod.gt_vcf, which lives in a different bucket
# and submission chain from ligated_vcf, and whether AnnotateTarget annotated
# that same object cannot be read off the table. So PHASED_VCF below is the
# workflow's own haplotag input, and the cell before check 5 compares phased
# genotypes between the two files directly instead of assuming they agree.
ANNOTATED_VCF = os.environ.get("ANNOTATED_VCF", "") or (
    f"{BUCKET}/submissions/5587dfff-6a06-46f3-97d2-28a738c308ce"
    "/PropagateFlareAncestry/9243c218-b1f2-47a3-8a6f-3e0ccc552ebf"
    "/call-AnnotateTarget/lai_prod.aou_lr_phase2_v1.chr20.ancestry.vcf.gz"
)

# The callset the workflow handed to whatshap. For the Array[File] runs that is
# the per-sample split VCF, not the multi-sample chromosome it was cut from --
# use the object the run actually consumed, or the check is vacuous. The split
# derives from flare_lai_prod.gt_vcf via `bcftools view -s` + `norm --rm-dup`,
# neither of which alters GT, so the frame question is unchanged; the file is
# just 37 MiB instead of 1.58 GiB.
#
# If you repoint the method config at ligated_vcf, repoint this too.
PHASED_VCF = os.environ.get("PHASED_VCF", "") or (
    f"{BUCKET}/refs/phased_split/{CHROM}/{SAMPLE}.phased.{CHROM}.vcf.gz"
)

# Six-population panel, in the shard order FlareByPopulation used.
ANC_LABELS = ["AFR", "AMR", "EAS", "EUR", "MID", "SAS"]
SV_BCF = "gs://prod-drc-broad/longreads/demo_group/integrated_sv_callset/v3_main.bcf"
SPLIT_BP = 50
APPLY_FILTERS = "PASS,."


def sh(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    return subprocess.run(cmd, shell=isinstance(cmd, str), check=True, text=True, **kw)


def cache_name(uri: str) -> str:
    """Local cache filename for a URI, namespaced by the WHOLE URI.

    Basenames collide across runs: every TrgtHaplotypePhase submission writes
    `<sample>.phase_summary.json`, `<sample>.hap_alleles.tsv.gz` and so on. A
    cache keyed on the basename therefore keeps serving the PREVIOUS run's
    outputs after RUN is repointed, with no download line and no error -- which
    is exactly what happened: checks 1-4 reported windowed numbers (14,618 loci,
    199,645 spanning reads) from a whole-genome submission (4,439,813 and
    56,337,861). Nothing in the output said so.
    """
    return f"{hashlib.sha1(uri.encode()).hexdigest()[:8]}_{Path(uri).name}"


def stage(uri: str) -> Path:
    if not uri.startswith("gs://"):
        return Path(uri).resolve()
    dest = WORK / cache_name(uri)
    if not dest.exists():
        partial = dest.with_suffix(dest.suffix + ".partial")
        sh(["gsutil", "-q", "cp", uri, str(partial)])
        partial.replace(dest)      # a half-copied object must not look cached
    return dest


def ls_one(pattern: str) -> str:
    out = subprocess.run(["gsutil", "ls", pattern], capture_output=True, text=True)
    hits = [l for l in out.stdout.splitlines() if l.startswith("gs://")]
    if not hits:
        raise FileNotFoundError(f"{pattern}\n{out.stderr.strip()}")
    return sorted(hits)[-1]


STAGE_MAX_GB = float(os.environ.get("STAGE_MAX_GB", "4"))


def gs_size_gb(uri: str) -> float:
    out = subprocess.run(["gsutil", "du", "-s", uri], capture_output=True, text=True)
    try:
        return int(out.stdout.split()[0]) / 1e9
    except (IndexError, ValueError):
        return float("inf")      # unknown size -> treat as too big to stage


def bcf_query(uri: str, fmt: str, sample: str = None, region: str = None,
              tag: str = None) -> Path:
    """`bcftools query` over a local or GCS object, cached to a TSV in WORK.

    htslib in these images has no GCS backend -- handing it a gs:// path fails
    with "Protocol not supported" -- so a remote object is either COPIED, when
    it is small enough to stage, or STREAMED through `gcloud storage cat` into
    bcftools on stdin. ANNOTATED_VCF is 28.98 GiB, which is why this exists:
    staging it is not an option and querying it directly does not work.

    Region filtering uses --targets, NOT -r, because -r needs an index and a
    pipe cannot have one. --targets filters while reading, so it behaves the
    same on a file and on a stream; the cost is that the whole object is read
    either way.

    The output is renamed into place only after the command succeeds, so an
    interrupted 29 GiB stream cannot leave a short file that later looks cached.
    """
    safe = re.sub(r"[^A-Za-z0-9._-]", "_", tag or cache_name(uri))
    dest = WORK / f"q_{safe}.tsv"
    if dest.exists():
        print(f"cached {dest}")
        return dest

    args = ["bcftools", "query", "-f", fmt]
    if sample:
        args += ["-s", sample]
    if region:
        args += ["--targets", region]

    tmp = dest.with_suffix(".partial")
    if uri.startswith("gs://") and gs_size_gb(uri) > STAGE_MAX_GB:
        print(f"$ gcloud storage cat {uri} | {' '.join(args)} -")
        with tmp.open("w") as fh:
            cat = subprocess.Popen(["gcloud", "storage", "cat", uri],
                                   stdout=subprocess.PIPE)
            q = subprocess.Popen(args + ["-"], stdin=cat.stdout, stdout=fh)
            cat.stdout.close()
            rc_q, rc_cat = q.wait(), cat.wait()
        if rc_q or rc_cat:
            tmp.unlink(missing_ok=True)
            raise RuntimeError(f"streaming {uri} failed "
                               f"(bcftools {rc_q}, gcloud storage cat {rc_cat})")
    else:
        src_path = stage(uri) if uri.startswith("gs://") else Path(uri)
        with tmp.open("w") as fh:
            sh(args + [str(src_path)], stdout=fh)
    tmp.replace(dest)
    return dest


print("window", REGION, "sample", SAMPLE)
print("RUN  ", RUN)

## 1. Tag census and read accounting

The allele-assignment tag in TRGT's spanning BAM is version-dependent, so the workflow never
guesses it. If this is the first run (`allele_tag` empty) the summary is census-only: read the tag
holding the allele index off the census below, then re-submit with it set.

`read_not_haplotagged` is the number to watch. Reads can only be haplotagged where the window holds
phased heterozygous SNVs, so a high count means thin phasing rather than a broken join.

In [ ]:
summary = json.loads(Path(stage(ls_one(f"{RUN}/**/{SAMPLE}.phase_summary.json"))).read_text())
accounting = pd.read_csv(stage(ls_one(f"{RUN}/**/{SAMPLE}.read_accounting.tsv")), sep="\t")

print("status:", summary["status"])
print("\nSAM tags in the spanning BAM (count over", summary["tag_census_records"], "records):")
for tag, n in summary["tag_census"].items():
    print(f"  {tag:>4}  {n:>9,}")

display(accounting)

if summary["status"] == "census_only":
    print("\nCensus-only run. Pick the allele-index tag above and re-submit with allele_tag set.")
else:
    reads = summary.get("reads", {})
    ht = summary.get("haplotag", {})
    total = max(reads.get("spanning_reads", 0), 1)
    print("\nread fates (of", f"{reads.get('spanning_reads', 0):,}", "spanning reads):")
    for key in ("read_joined", "read_not_haplotagged", "read_no_allele_tag",
                "read_no_locus", "read_ambiguous_locus", "read_allele_tag_unparsed"):
        if key in reads:
            print(f"  {key:>28}  {reads[key]:>9,}  ({100 * reads[key] / total:5.1f}%)")
    print("\nhaplotag list:", f"{summary.get('haplotag_reads', 0):,} tagged;",
          f"{ht.get('read_untagged', 0):,} untagged")

## 2. Resolution tally

`homozygous` loci are not a success and not a failure: both slots hold the same allele, so there is
nothing to orient. The number that matters is `reads` as a share of the loci where orientation
*could* change something — `reads + conflict + insufficient_reads`.

`conflict` means both alleles voted for the same haplotype. That is the diagnostic worth watching:
a few are expected from mismapped reads, many would mean the haplotagging is not doing its job.

In [ ]:
assert summary["status"] == "ok", "re-run with allele_tag set before running the rest"

res = pd.Series(summary["resolution"]).rename("loci").to_frame()
res["share_of_loci"] = (res["loci"] / summary["loci"]).round(4)
display(res)

print(f"catalog loci in window:        {summary['loci']:,}")
print(f"orientation-relevant loci:     {summary['n_orientation_relevant']:,}")
frac = summary["frac_resolved_of_relevant"]
print(f"resolved from reads:           {frac:.1%}" if frac is not None else "resolved: n/a")

alleles = pd.read_csv(stage(ls_one(f"{RUN}/**/{SAMPLE}.hap_alleles.tsv.gz")), sep="\t")
print("\nper-haplotype rows:", f"{len(alleles):,}", "| loci:", f"{alleles.locus_id.nunique():,}")
display(alleles.head())

read_backed = alleles[alleles.phase_source == "reads"]
if len(read_backed):
    print("\nsupporting reads per oriented haplotype:")
    display(read_backed.n_reads.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2))
    print("purity below 1.0:", f"{(read_backed.purity < 1).mean():.1%} of oriented haplotypes")

## 3. Did re-genotyping move the calls?

The workflow re-runs TRGT on haplotagged reads, so its genotypes need not be identical to the
as-delivered ones — TRGT has used `HP` during genotyping since 0.7.0, which is a deliberate change
in how the alleles are called.

Compare the **unordered** allele-length pairs. A mismatch is a genotyping difference, not a phasing
one. Stratify by the larger length change so a disagreement concentrated at long expansions can be
told from one spread across the whole range.

In [ ]:
QUERY = "%CHROM\t%POS\t%INFO/END\t%INFO/TRID\t%INFO/MOTIFS\t[%GT]\t[%AL]\n"


def trgt_frame(path: str, label: str) -> pd.DataFrame:
    out = subprocess.run(["bcftools", "query", "-f", QUERY, str(path)],
                         capture_output=True, text=True)
    assert out.returncode == 0, out.stderr
    rows = []
    for line in out.stdout.splitlines():
        chrom, pos, end, trid, motifs, gt, al = line.split("\t")
        lengths = tuple(sorted(int(v) for v in al.split(",") if v.strip().isdigit()))
        rows.append({
            "locus_id": trid.split(",")[0],
            "chrom": chrom,
            "pos": int(pos),
            "ref_len": int(end) - int(pos) + 1,
            f"gt_{label}": gt,
            f"pair_{label}": lengths,
            # kept for check 3a: the raw TRID carries the catalog's own
            # coordinates and motif, so it can be checked against the record
            f"trid_raw_{label}": trid,
            f"motifs_{label}": motifs,
        })
    return pd.DataFrame(rows)


regen = trgt_frame(stage(ls_one(f"{RUN}/**/{SAMPLE}.trgt.vcf.gz")), "regen")
prod = trgt_frame(stage(ls_one(f"{RUN}/**/{SAMPLE}.production_trgt.vcf.gz")), "prod")

both = regen.merge(prod[["locus_id", "gt_prod", "pair_prod", "motifs_prod"]],
                   on="locus_id", how="inner")
print(f"loci re-genotyped: {len(regen):,} | in production slice: {len(prod):,} | shared: {len(both):,}")

both["same_pair"] = both.pair_regen == both.pair_prod
both["max_change"] = [
    max((abs(v - r) for v in pair), default=0)
    for pair, r in zip(both.pair_regen, both.ref_len)
]
bins = [-1, 0, 19, 49, np.inf]
labels = ["0 bp", "1-19 bp", "20-49 bp", ">=50 bp"]
both["stratum"] = pd.cut(both.max_change, bins=bins, labels=labels)

tab = both.groupby("stratum", observed=False).agg(
    loci=("same_pair", "size"), same_pair=("same_pair", "sum"))
tab["agreement"] = (tab.same_pair / tab.loci.replace(0, np.nan)).round(4)
display(tab)
print(f"\noverall unordered-pair agreement: {both.same_pair.mean():.4f}")

### 3a. Where does the disagreement live?

Genome-wide the re-genotyped and production calls agree on only 93.63% of loci, against 98.17% in the
chr20 window — so the window was not representative and the "re-genotyping barely moves the calls"
reading of check 3 does not survive at scale. 6.37% is ~283,000 loci.

The reads are the same reads and TRGT ignores `HP`, so a difference of that size has to come from the
*definitions*. Ours were reconstructed from the production VCF by `trgt_defs_from_vcf.py`, and the
measured TRID agreement against the real catalog was 0.9729 on start and 0.9499 on span — a ~5% span
mismatch sitting suspiciously close to a 6.37% call disagreement.

A TRID encodes the catalog's own `chrom-start-end-motif`, so each record carries an independent copy
of what the locus was *supposed* to be. If disagreement concentrates where the TRID-implied extent or
motif contradicts the record, the cause is our derived definitions — fixable, and it would mean the
production calls are the better ones. If it does not, the explanation is elsewhere and the comparison
itself needs rethinking before any of it is trusted.

Compound loci are compared against the union of their components, not the first one. The catalog
contains them (`20-10002373-10002383-GTG` sits in a record spanning 10,002,373–10,002,525), and
testing the first component alone would flag every compound locus as broken and manufacture a
definitions problem that isn't there.


In [ ]:
def trid_extent(trid_raw: str):
    """(min start, max end, motifs, n components) a TRID claims, or Nones.

    A TRID is the catalog's own record of the locus, so it is independent
    evidence of what the definition should have been. Two traps:

    * Split each component from the RIGHT -- contig names may contain hyphens,
      the trailing three fields may not.
    * A COMPOUND locus carries several comma-separated components and the
      record's [POS, END] is their UNION, so the span must be compared against
      min(start)..max(end), not against the first component. The catalog really
      does contain these -- 20-10002373-10002383-GTG sits in a record spanning
      10,002,373-10,002,525, span 153 against the component's 11. Testing the
      first component alone would flag every compound locus as inconsistent and
      invent a definitions problem that is not there.
    """
    starts, ends, motifs = [], [], []
    for part in trid_raw.split(","):
        bits = part.rsplit("-", 3)
        if len(bits) != 4 or not (bits[1].isdigit() and bits[2].isdigit()):
            return None, None, None, 0
        starts.append(int(bits[1]))
        ends.append(int(bits[2]))
        motifs.append(bits[3])
    return min(starts), max(ends), motifs, len(starts)


ext = [trid_extent(x) for x in both.trid_raw_regen]
both["trid_start"] = [e[0] for e in ext]
both["trid_end"] = [e[1] for e in ext]
both["trid_motifs"] = [e[2] for e in ext]
both["n_components"] = [e[3] for e in ext]
parsed = both.trid_start.notna()
print(f"TRIDs parsed: {parsed.sum():,} of {len(both):,} ({parsed.mean():.4f})")
print(f"compound loci (>1 component): {(both.n_components > 1).sum():,}"
      f" ({(both.n_components > 1).mean():.4f})")

ok = both[parsed].copy()
ok["start_ok"] = ok.trid_start == ok.pos
ok["span_ok"] = (ok.trid_end - ok.trid_start + 1) == ok.ref_len
ok["motif_ok"] = [list(ms) == str(rec).split(",")
                  for ms, rec in zip(ok.trid_motifs, ok.motifs_regen)]
ok["defs_ok"] = ok.start_ok & ok.span_ok & ok.motif_ok

print("\nhow often the record reproduces what its own TRID claims:")
for col in ("start_ok", "span_ok", "motif_ok", "defs_ok"):
    print(f"  {col:9s} {ok[col].mean():.4f}")

print("\nagreement with production, split by definition self-consistency:")
by = ok.groupby("defs_ok", observed=False).agg(
    loci=("same_pair", "size"), same_pair=("same_pair", "sum"))
by["agreement"] = (by.same_pair / by.loci.replace(0, np.nan)).round(4)
display(by)

# The number that decides it: of the loci that disagree, how many already sit
# on a definition contradicting its own TRID? Compare against the base rate --
# a share that merely matches the base rate explains nothing.
dis = ~ok.same_pair
if dis.sum():
    share = (dis & ~ok.defs_ok).sum() / dis.sum()
    base = (~ok.defs_ok).mean()
    print(f"\ndisagreeing loci: {dis.sum():,}")
    print(f"  definition self-inconsistent: {(dis & ~ok.defs_ok).sum():,} ({share:.1%})")
    print(f"  base rate of self-inconsistency: {base:.1%}")
    print(f"  enrichment: {share / base:.2f}x" if base else "")
    print("\nFar above the base rate -> the derived definitions are the cause, and"
          "\nthe production calls are the better ones. At the base rate -> they are"
          "\nnot, and the explanation lies elsewhere.")

print("\nagreement by which component disagrees:")
display(ok.groupby(["start_ok", "span_ok", "motif_ok"], observed=False).agg(
    loci=("same_pair", "size"), agreement=("same_pair", "mean")).round(4))

print("\nagreement by locus complexity:")
display(ok.assign(kind=np.where(ok.n_components > 1, "compound", "simple"))
          .groupby("kind", observed=False)
          .agg(loci=("same_pair", "size"), agreement=("same_pair", "mean")).round(4))


## 4. Orientation against the integrated callset

Now the comparison that motivated the work. `aggregate_repeat_loci.py` gives the integrated
callset's own per-haplotype length changes over the same window, with `AN1`/`AN2` attached. For each
length-heterozygous locus, ask whether it puts the longer allele on the same haplotype the read-level
join did.

Expect agreement to track the concordance table: high at 0 bp and 1–19 bp, falling at ≥ 50 bp. The
fall is the point. It is where summing GLnexus indels and merged SV records double-counts, and the
reason the dosage arm is being re-based.

In [ ]:
samples_txt = WORK / "samples.txt"
samples_txt.write_text(SAMPLE + "\n")
locus_vcf = WORK / "integrated.loci.vcf"

if not locus_vcf.exists():
    sh([sys.executable, str(_scripts / "aggregate_repeat_loci.py"),
        "--small-vcf", ANNOTATED_VCF,
        "--sv-vcf", SV_BCF,
        "--split-bp", str(SPLIT_BP),
        "--apply-filters", APPLY_FILTERS,
        "--catalog-bed", str(stage(CATALOG)),
        "--region", REGION,
        "--samples", str(samples_txt),
        "--with-ancestry",
        "--out-vcf", str(locus_vcf),
        "--out-alleles", str(WORK / "integrated.alleles.tsv.gz"),
        "--out-summary", str(WORK / "integrated.summary.json")])

agg_summary = json.loads((WORK / "integrated.summary.json").read_text())
print("integrated-callset haplotype QC:")
for key in ("hap_missing_frac", "hap_possible_duplicate_frac", "hap_ancestry_inconsistent_frac"):
    if key in agg_summary:
        print(f"  {key:>34}  {agg_summary[key]:.4f}")

In [ ]:
INT_QUERY = "%CHROM\t%POS\t%ID\t%INFO/RU_DOSAGE\t%INFO/PERIOD\t[%GT]\t[%AN1]\t[%AN2]\n"


def integrated_haplotypes(path: Path) -> pd.DataFrame:
    """Per-haplotype signed bp change and ancestry, from the locus VCF."""
    out = subprocess.run(["bcftools", "query", "-f", INT_QUERY, str(path)],
                         capture_output=True, text=True)
    assert out.returncode == 0, out.stderr
    rows = []
    for line in out.stdout.splitlines():
        chrom, pos, locus_id, dosage, period, gt, an1, an2 = line.split("\t")
        if "|" not in gt:          # unphased locus: haplotype order is not meaningful
            continue
        try:
            per = int(period)
        except ValueError:
            per = 1
        units = [0.0] + [float(v) for v in dosage.split(",") if v not in (".", "")]
        idx = [int(a) if a.isdigit() else None for a in gt.split("|")]
        if any(i is None or i >= len(units) for i in idx):
            continue
        rows.append({
            "locus_id": locus_id,
            "chrom": chrom,
            "pos": int(pos),
            "int_hap1_bp": units[idx[0]] * per,
            "int_hap2_bp": units[idx[1]] * per,
            "an1": an1,
            "an2": an2,
        })
    return pd.DataFrame(rows)


integrated = integrated_haplotypes(locus_vcf)
print("integrated loci with phased genotypes:", f"{len(integrated):,}")

# TRGT side: signed bp change per haplotype, oriented by the read-level join.
trgt_wide = (alleles[alleles.phase_source == "reads"]
             .pivot_table(index="locus_id", columns="hap", values="change_bp", aggfunc="first")
             .rename(columns={1: "trgt_hap1_bp", 2: "trgt_hap2_bp"})
             .dropna()
             .reset_index())
print("TRGT loci oriented from reads:", f"{len(trgt_wide):,}")

cmp = trgt_wide.merge(integrated, on="locus_id", how="inner")
print("comparable loci:", f"{len(cmp):,}")

# Only length-heterozygous loci carry orientation information on either side.
cmp["trgt_het"] = cmp.trgt_hap1_bp != cmp.trgt_hap2_bp
cmp["int_het"] = cmp.int_hap1_bp != cmp.int_hap2_bp
usable = cmp[cmp.trgt_het & cmp.int_het].copy()

usable["agree"] = (
    (usable.trgt_hap1_bp > usable.trgt_hap2_bp) == (usable.int_hap1_bp > usable.int_hap2_bp)
)
usable["trgt_max_change"] = usable[["trgt_hap1_bp", "trgt_hap2_bp"]].abs().max(axis=1)
usable["stratum"] = pd.cut(usable.trgt_max_change, bins=bins, labels=labels)

tab4 = usable.groupby("stratum", observed=False).agg(
    loci=("agree", "size"), agree=("agree", "sum"))
tab4["orientation_agreement"] = (tab4.agree / tab4.loci.replace(0, np.nan)).round(4)
display(tab4)
print(f"\nloci informative on both sides: {len(usable):,}"
      f" | overall orientation agreement: {usable.agree.mean():.4f}"
      if len(usable) else "no locus is length-heterozygous on both sides")

### 5a. Does the ancestry VCF share the haplotype frame we tagged in?

Check 5 reads `AN1` as the ancestry of haplotype 1 — the haplotype whatshap assigned. That holds only
if the annotated VCF carries the *same phased genotypes* as the callset the workflow haplotagged
against. It is tempting to argue this from provenance, since `PropagateFlareAncestry` annotates the
FLARE target, but the table does not support that argument: the workflow was configured with
`gt_vcf`, which sits in a different bucket and submission chain from `ligated_vcf`, and nothing in
the row says which object `AnnotateTarget` consumed.

So compare the genotypes. If the phased `GT` strings agree at shared sites, the two files share a
haplotype frame and `hap1 <-> AN1` is sound. If they disagree, every per-ancestry number downstream
is assigning lengths to the wrong ancestry, and the exposure count is worse than useless.


In [ ]:
def phased_gt(uri: str, name: str) -> pd.DataFrame:
    path = bcf_query(uri, "%POS\t[%GT]\n", sample=SAMPLE, region=REGION,
                     tag=f"gt_{name}")
    df = pd.read_csv(path, sep="\t", names=["pos", f"gt_{name}"],
                     dtype={f"gt_{name}": str})
    return df.drop_duplicates("pos")


frame = (phased_gt(PHASED_VCF, "hap")
         .merge(phased_gt(ANNOTATED_VCF, "anc"), on="pos", how="inner"))
print(f"sites shared by the haplotag callset and the annotated VCF: {len(frame):,}")
assert len(frame), "no shared sites -- PHASED_VCF and ANNOTATED_VCF do not overlap in this window"

both_phased = frame[frame.gt_hap.str.contains(r"\|") & frame.gt_anc.str.contains(r"\|")]
identical = (both_phased.gt_hap == both_phased.gt_anc)
het = both_phased[both_phased.gt_hap.isin(["0|1", "1|0"])]
het_identical = (het.gt_hap == het.gt_anc)

print(f"phased in both:            {len(both_phased):,}")
print(f"identical phased GT:       {identical.mean():.6f}" if len(both_phased) else "")
print(f"heterozygous sites:        {len(het):,}")
print(f"  identical at hets:       {het_identical.mean():.6f}" if len(het) else "")

# Hets are the only sites that carry orientation; a frame flip shows up there as
# agreement near 0, and a frame match as agreement near 1. Anything in between
# means the two files are phased independently and AN1 cannot be trusted as hap1.
PHASE_FRAME_OK = len(het) > 0 and het_identical.mean() > 0.99
if not PHASE_FRAME_OK and len(het):
    print(f"\nFRAME MISMATCH: hets agree at {het_identical.mean():.1%}."
          " Do not read check 5's exposure count -- AN1/AN2 are not in the"
          " haplotype order whatshap assigned.")


## 5. Exposure — where orientation actually changes a dosage

A swap only matters when the two haplotypes carry *different* local ancestry. If both are ancestry
*k*, swapping their lengths leaves `DS{k}` unchanged, because it is a sum over one bin. So the
person-loci at risk are the length-heterozygous **and** ancestry-discordant ones.

This is the number that sizes the genome-wide job. If it is small, the per-ancestry length test is
machinery built to recover a claim the data barely supports, and a pooled length test plus a
per-ancestry carrier test is the better design.

Two deliberate choices in what follows.

**Ancestry is read straight from the annotated VCF, not from `aggregate_repeat_loci.py`.** Check 4
necessarily goes through that script, because comparing against the integrated callset is the whole
point of it. Check 5 must not: its length inference is the defect under review, so restricting the
denominator to loci where it also produced a phased call would make the exposure count a function of
the thing we are trying to replace. `AN1`/`AN2` come from FLARE and are independent of it.

**Ancestry at a TR locus comes from the flanking markers.** Local ancestry is piecewise constant, so
a locus inherits what the markers either side of it agree on. Where they disagree the locus straddles
an ancestry switch, and it is left unassigned rather than guessed — switches are rare, so this should
cost almost nothing, and `ancestry_assigned` below reports what it actually cost.


In [ ]:
assert PHASE_FRAME_OK, (
    "phase frames disagree (cell 5a) -- AN1 is not haplotype 1 of the frame whatshap"
    " tagged in, so every per-ancestry assignment below would be wrong")

# Ancestry markers for this sample over the window, straight from the annotated VCF.
#
# Through bcf_query, not a direct bcftools call: ANNOTATED_VCF is a gs:// object
# and htslib here has no GCS backend, and -r needs an index a stream cannot
# have. This cell previously rolled its own cache under a FIXED filename, which
# meant a stale or empty ancestry_markers.tsv was read in preference to
# re-querying -- the failure then surfaced as "no usable AN1/AN2", blaming the
# input rather than the cache.
ANC_QUERY = "%POS\t[%AN1]\t[%AN2]\n"
anc_path = bcf_query(ANNOTATED_VCF, ANC_QUERY, sample=SAMPLE, region=REGION,
                     tag="ancestry_markers")

mk = pd.read_csv(anc_path, sep="\t", names=["pos", "an1", "an2"],
                 dtype={"an1": str, "an2": str})
mk = mk[mk.an1.str.isdigit() & mk.an2.str.isdigit()].sort_values("pos").reset_index(drop=True)
assert len(mk), "no usable AN1/AN2 -- is ANNOTATED_VCF the PropagateFlareAncestry output?"
print(f"ancestry markers in window: {len(mk):,}"
      f" | hap1 switches: {int((mk.an1 != mk.an1.shift()).sum()) - 1}"
      f" | hap2 switches: {int((mk.an2 != mk.an2.shift()).sum()) - 1}")
seen = sorted(set(mk.an1) | set(mk.an2), key=int)
print("ancestry indices present:", seen,
      "->", [ANC_LABELS[int(i)] if int(i) < len(ANC_LABELS) else f"?{i}" for i in seen])
assert all(int(i) < len(ANC_LABELS) for i in seen), (
    f"ancestry index outside the {len(ANC_LABELS)}-population panel {ANC_LABELS}"
    " -- the AN1/AN2 encoding is not what this cell assumes")


def ancestry_at(loci_pos: np.ndarray) -> pd.DataFrame:
    """AN1/AN2 at each locus, taken from the flanking markers.

    Local ancestry is piecewise constant, so a locus inherits what its neighbours
    agree on. Where the flanks disagree the locus straddles an ancestry switch and
    is left unassigned ("") rather than guessed.
    """
    p = mk.pos.to_numpy()
    hi = np.clip(np.searchsorted(p, loci_pos, side="left"), 0, len(p) - 1)
    lo = np.clip(hi - 1, 0, len(p) - 1)
    cols = {}
    for name in ("an1", "an2"):
        a = mk[name].to_numpy()
        cols[name] = np.where(a[lo] == a[hi], a[hi], "")
    return pd.DataFrame(cols)


# The loci the read-level join actually oriented: the only ones where a swap is
# ours to get wrong. Homozygous loci are excluded by construction.
oriented = (alleles[alleles.phase_source == "reads"]
            .pivot_table(index="locus_id", columns="hap", values="change_bp", aggfunc="first")
            .rename(columns={1: "hap1_bp", 2: "hap2_bp"})
            .dropna()
            .reset_index()
            .merge(regen[["locus_id", "pos"]], on="locus_id", how="inner")
            .reset_index(drop=True))
oriented = pd.concat([oriented, ancestry_at(oriented.pos.to_numpy())], axis=1)

oriented["length_het"] = oriented.hap1_bp != oriented.hap2_bp
oriented["assigned"] = oriented.an1.ne("") & oriented.an2.ne("")
oriented["anc_discordant"] = oriented.assigned & (oriented.an1 != oriented.an2)
oriented["exposed"] = oriented.length_het & oriented.anc_discordant

exposure = pd.DataFrame({
    "oriented_loci": [len(oriented)],
    "ancestry_assigned": [int(oriented.assigned.sum())],
    "length_het": [int(oriented.length_het.sum())],
    "anc_discordant": [int(oriented.anc_discordant.sum())],
    "exposed": [int(oriented.exposed.sum())],
})
exposure["frac_of_oriented"] = (
    exposure.exposed / exposure.oriented_loci.replace(0, np.nan)).round(4)
exposure["frac_of_window_loci"] = (exposure.exposed / summary["loci"]).round(5)
display(exposure)

n_exposed = int(oriented.exposed.sum())
print(f"Of {summary['loci']:,} catalog loci in the window, {n_exposed:,} are both"
      " length-heterozygous and ancestry-discordant for this sample.")

# Scale by catalog size rather than by bp: locus density is not uniform.
catalog_loci = sum(1 for _ in gzip.open(stage(CATALOG), "rt"))
scale = catalog_loci / summary["loci"]
print(f"\nCatalog is {catalog_loci:,} loci, {scale:.1f}x this window, so genome-wide this"
      f" sample would carry roughly {n_exposed * scale:,.0f} exposed loci.")
print("Multiply by the cohort only after checking that this sample's ancestry switch count"
      " is typical -- a single-ancestry participant has an exposure of zero by definition,"
      " and the cohort mean is what sizes the job.")

if n_exposed:
    print("\nexposed loci by ancestry pair:")
    def label(i):
        return ANC_LABELS[int(i)]

    pair = (oriented[oriented.exposed]
            .assign(pair=lambda d: [
                "/".join(sorted((label(a), label(b))))
                for a, b in zip(d.an1, d.an2)])
            .pair.value_counts().rename("loci").to_frame())
    display(pair)


## Verdict

Fill this in from the cells above before anyone writes a genome-wide version.

| Question | Where | Pass looks like |
|---|---|---|
| Allele tag identified | §1 census | one integer-valued tag present on ~every spanning read |
| Reads actually haplotagged | §1 `read_not_haplotagged` | a small minority |
| Orientation resolved | §2 `frac_resolved_of_relevant` | high, with few `conflict` |
| Re-genotyping stable | §3 | agreement near 1 at 0 bp; any drop understood before going on |
| Orientation beats summing | §4 | high at 0–19 bp, falling at ≥ 50 bp |
| Worth doing at all | §5 `frac_exposed` | large enough to justify the read pass |

A failure in §1 or §2 is a plumbing problem. A surprise in §3 is a genotyping question and has to be
settled first. §4 behaving as predicted is the result this prototype exists to produce; §4 agreeing
*everywhere*, including at ≥ 50 bp, would mean the double-counting does not reach orientation and
the integrated callset's own ordering would have been good enough.